In [3]:
import math
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import spikegen
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

In [4]:
def print_batch_accuracy(data, targets, train=False):
    output, _, _, _ = net(data.view(batch_size, -1))
    _, idx = output.sum(dim=0).max(1)
    acc = np.mean((targets == idx).detach().cpu().numpy())

    if train:
        print(f"Train set accuracy for a single minibatch: {acc*100:.2f}%")
    else:
        print(f"Test set accuracy for a single minibatch: {acc*100:.2f}%")

def train_printer(
    data, targets, epoch,
    counter, iter_counter,
        loss_hist, test_loss_hist, test_data, test_targets):
    print(f"Epoch {epoch}, Iteration {iter_counter}")
    print(f"Train Set Loss: {loss_hist[counter]:.2f}")
    print(f"Test Set Loss: {test_loss_hist[counter]:.2f}")
    print_batch_accuracy(data, targets, train=True)
    print_batch_accuracy(test_data, test_targets, train=False)
    print("\n")

In [5]:
batch_size = 128
data_path='/tmp/data/mnist'

dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
## DATA PREP
# Define a transform
transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.Grayscale(),
            transforms.ToTensor(),
            transforms.Normalize((0,), (1,))])

mnist_train = datasets.MNIST(data_path, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(data_path, train=False, download=True, transform=transform)
# Create DataLoaders
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=True, drop_last=True)

In [6]:
class STDPPlasticity:
    """
    Pair-based STDP with decaying pre/post traces.
    Updates a given weight matrix W in-place (shape: [out_features, in_features]).
    """
    def __init__(
        self,
        weight: torch.Tensor,
        tau_pre: float = 20.0,
        tau_post: float = 20.0,
        A_plus: float = 0.020,
        A_minus: float = 0.021,
        lr: float = 1e-4,
        wmin: float | None = None,
        wmax: float | None = None,
        dt: float = 1.0, # = time pre_synap spike - time pos_synap spike
    ):
        self.W = weight
        self.A_plus = A_plus
        self.A_minus = A_minus
        self.lr = lr
        self.wmin = wmin
        self.wmax = wmax

        self.decay_pre = math.exp( -dt / tau_pre)
        self.decay_post = math.exp( -dt / tau_post)

        self.pre_trace = None    # (B, in_features)
        self.post_trace = None   # (B, out_features)

    def reset_state(self):
        self.pre_trace = None
        self.post_trace = None

    @torch.no_grad()
    def step(self, pre_spk: torch.Tensor, post_spk: torch.Tensor):
        """
        pre_spk:  (Batch_size(B), in_features) spikes (0/1 or float in [0,1])
        post_spk: (Batch_size(B), out_features)
        we do the no_grad here because weights are manually calculated
        this is used to calculate the weight changes
        """
        B, in_features = pre_spk.shape
        _, out_features = post_spk.shape

        device = pre_spk.device
        dtype = self.W.dtype

        if self.pre_trace is None:
            self.pre_trace = torch.zeros((B, in_features), device=device, dtype=dtype)
            self.post_trace = torch.zeros((B, out_features), device=device, dtype=dtype)

        pre = pre_spk.to(dtype)
        post = post_spk.to(dtype)

        # Decay traces + add current spikes
        self.pre_trace.mul_(self.decay_pre).add_(pre)
        self.post_trace.mul_(self.decay_post).add_(post)

        # STDP correlations
        pot = post.T @ self.pre_trace            # (out, in)
        dep = self.post_trace.T @ pre            # (out, in)
        # devided by B cause this is batch calculation
        dW = (self.A_plus * pot - self.A_minus * dep) / B
        self.W.add_(self.lr * dW)

        if self.wmin is not None or self.wmax is not None:
            lo = self.wmin if self.wmin is not None else -float("inf")
            hi = self.wmax if self.wmax is not None else float("inf")
            self.W.clamp_(lo, hi)


class SNN_STDP(nn.Module):
    """
    * spk_out_rec: (T,B,num_outputs)
    * mem_out_rec: (T,B,num_outputs)
    * spk_pre_output_rec: (T,B,hidden_num)  # pre-synaptic spikes into outputFC
    * spk_in_rec: (T,B,num_inputs)          # optional, if you want STDP on inputFC
      - includes helper to apply STDP to chosen layers after forward (recommended)
    """
    def __init__(
        self,
        layer_num,
        beta,
        step_num,
        hidden_num,
        num_inputs,
        num_outputs,
        stdp_lr=1e-4,
        tau_pre=20.0,
        tau_post=20.0,
        A_plus=0.010,
        A_minus=0.012,
        wmin=None,
        wmax=None,
    ):
        super().__init__()
        self.layer_num = layer_num
        self.beta = beta
        self.step_num = step_num
        self.hidden_num = hidden_num

        # Layers
        self.inputFC = nn.Linear(num_inputs, self.hidden_num)
        self.inputLIF = snn.Leaky(beta=self.beta)

        self.innerLayer = nn.ModuleList()
        self.innerLIF = nn.ModuleList()
        for _ in range(layer_num):
            self.innerLayer.append(nn.Linear(self.hidden_num, self.hidden_num))
            self.innerLIF.append(snn.Leaky(beta=self.beta))

        self.outputFC = nn.Linear(self.hidden_num, num_outputs)
        self.outputLIF = snn.Leaky(beta=self.beta)

        # trach the output weights
        self.stdp_output = STDPPlasticity(
            self.outputFC.weight,
            tau_pre=tau_pre,
            tau_post=tau_post,
            A_plus=A_plus,
            A_minus=A_minus,
            lr=stdp_lr,
            wmin=wmin,
            wmax=wmax,
        )

        # track the input weights
        self.stdp_input = STDPPlasticity(
            self.inputFC.weight,
            tau_pre=tau_pre,
            tau_post=tau_post,
            A_plus=A_plus,
            A_minus=A_minus,
            lr=stdp_lr,
            wmin=wmin,
            wmax=wmax,
        )

    def reset_stdp_state(self):
        """Call once per batch/sequence if you want traces reset (matches your training loop style)."""
        self.stdp_output.reset_state()
        self.stdp_input.reset_state()

    # def apply_stdp(self, spk_in_rec, spk_pre_output_rec, spk_out_rec, apply_to_input=False):
    #     """
    #     Apply STDP *after* forward (and typically after optimizer.step()).

    #     spk_in_rec:          (T,B,num_inputs)
    #     spk_pre_output_rec:  (T,B,hidden_num)
    #     spk_out_rec:         (T,B,num_outputs)
    #     """
    #     with torch.no_grad():
    #         # outputFC: pre = hidden spikes, post = output spikes
    #         for t in range(spk_out_rec.size(0)):
    #             self.stdp_output.step(spk_pre_output_rec[t], spk_out_rec[t])

    #         # inputFC: pre = input spikes (if your x is already spikes), post = first hidden spikes
    #         # NOTE: in your current network, x is *static* over time, not spikes.
    #         # If you rate-encode x into spikes over time, then spk_in_rec makes sense for STDP.
    #         if apply_to_input:
    #             # post for inputFC would be spikes after inputLIF; we don't currently return that,
    #             # so you would need to record it too (see comments below).
    #             raise NotImplementedError(
    #                 "To STDP-train inputFC, you must provide post spikes from inputLIF each timestep."
    #             )

    def forward(self, x):
        """
        x: (B, num_inputs)  (your code uses static input across time)

        Returns:
          spk_out_rec:         (T,B,num_outputs)
          mem_out_rec:         (T,B,num_outputs)
          spk_pre_output_rec:  (T,B,hidden_num)   # spikes right before outputFC
          spk_in_rec:          (T,B,num_inputs)   # repeated static input for completeness
        """
        # Initialize hidden states at t=0
        mem_in = self.inputLIF.init_leaky()
        mem_hidden = [layer.init_leaky() for layer in self.innerLIF]
        mem_out = self.outputLIF.init_leaky()

        # Recordings
        spk_out_rec = []
        mem_out_rec = []
        spk_pre_output_rec = []
        spk_in_rec = []

        for _ in range(self.step_num):
            # input is static each timestep (not spikes)
            spk_in_rec.append(x)

            # Input -> first spiking layer
            cur = self.inputFC(x)
            spk, mem_in = self.inputLIF(cur, mem_in)

            # Inner layers
            for idx in range(self.layer_num):
                cur = self.innerLayer[idx](spk)
                spk, mem_hidden[idx] = self.innerLIF[idx](cur, mem_hidden[idx])

            # spk here is the pre-synaptic spikes into outputFC
            spk_pre_output_rec.append(spk)

            # Output layer
            cur = self.outputFC(spk)
            spk_out, mem_out = self.outputLIF(cur, mem_out)

            spk_out_rec.append(spk_out)
            mem_out_rec.append(mem_out)

        return (
            torch.stack(spk_out_rec, dim=0),
            torch.stack(mem_out_rec, dim=0),
            torch.stack(spk_pre_output_rec, dim=0),
            torch.stack(spk_in_rec, dim=0),
        )

In [8]:
# betas = [0.75, 0.85, 0.95, 0.99]
# layer_nums = [2, 3, 4, 5]
# neuron_nums = [100, 200]
# step_nums = [25, 50, 100]
betas = [0.99]
layer_nums = [5]
neuron_nums = [100]
step_nums = [20]

accuracy_rec = []

for b in betas:
  for sn in step_nums:
    for ln in layer_nums:
      for n in neuron_nums:

        net = SNN_STDP(
            layer_num=ln, beta=b, step_num=sn, hidden_num=n,
            num_inputs=28*28, num_outputs=10,
            stdp_lr=1e-4
        ).to(device)

        loss_fn = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(net.parameters(), lr=5e-4, betas=(0.9, 0.999))

        num_epochs = 1
        loss_hist = []
        test_loss_hist = []
        counter = 0

        # Outer training loop
        for epoch in range(num_epochs):
          iter_counter = 0

          # Minibatch training loop
          for data, targets in train_loader:
            data = data.to(device)
            targets = targets.to(device)

            # forward pass
            net.train()
            spk_out_rec, mem_out_rec, spk_pre_out_rec, spk_in_rec = net(data.view(data.size(0), -1))

            # loss summed over time
            loss_val = torch.zeros((1), dtype=dtype, device=device)
            for step in range(sn):
              loss_val += loss_fn(mem_out_rec[step], targets)

            # supervised update
            optimizer.zero_grad(set_to_none=True)
            loss_val.backward()
            optimizer.step()

            # STDP update (local plasticity)
            net.reset_stdp_state()
            with torch.no_grad():
              # Apply STDP to outputFC: pre=hidden spikes, post=output spikes
              for step in range(sn):
                net.stdp_output.step(spk_pre_out_rec[step], spk_out_rec[step])

            # Store loss history for future plotting
            loss_hist.append(loss_val.item())

            # Test minibatch
            with torch.no_grad():
              net.eval()
              test_data, test_targets = next(iter(test_loader))
              test_data = test_data.to(device)
              test_targets = test_targets.to(device)

              test_spk_out, test_mem_out, _, _ = net(test_data.view(test_data.size(0), -1))

              test_loss = torch.zeros((1), dtype=dtype, device=device)
              for step in range(sn):
                test_loss += loss_fn(test_mem_out[step], test_targets)
              test_loss_hist.append(test_loss.item())

              if counter % 50 == 0:
                train_printer(
                  data, targets, epoch,
                  counter, iter_counter,
                  loss_hist, test_loss_hist,
                  test_data, test_targets
                )

              counter += 1
              iter_counter += 1

        #Final full test accuracy
        total = 0
        correct = 0

        # drop_last switched to False to keep all samples
        test_loader_full = DataLoader(mnist_test, batch_size=batch_size, shuffle=True, drop_last=False)

        with torch.no_grad():
          net.eval()
          for data, targets in test_loader_full:
            data = data.to(device)
            targets = targets.to(device)

            test_spk_out, _, _, _ = net(data.view(data.size(0), -1))

            # spike-count readout
            _, predicted = test_spk_out.sum(dim=0).max(1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

        print(f"Total correctly classified test set images: {correct}/{total}")
        print(f"Test Set Accuracy: {100 * correct / total:.2f}%")
        accuracy_rec.append(100 * correct / total)

Epoch 0, Iteration 0
Train Set Loss: 50.31
Test Set Loss: 48.55
Train set accuracy for a single minibatch: 10.94%
Test set accuracy for a single minibatch: 9.38%


Epoch 0, Iteration 50
Train Set Loss: 39.58
Test Set Loss: 39.92
Train set accuracy for a single minibatch: 26.56%
Test set accuracy for a single minibatch: 26.56%


Epoch 0, Iteration 100
Train Set Loss: 26.32
Test Set Loss: 24.22
Train set accuracy for a single minibatch: 67.19%
Test set accuracy for a single minibatch: 70.31%


Epoch 0, Iteration 150
Train Set Loss: 16.76
Test Set Loss: 19.45
Train set accuracy for a single minibatch: 85.94%
Test set accuracy for a single minibatch: 83.59%


Epoch 0, Iteration 200
Train Set Loss: 11.79
Test Set Loss: 13.35
Train set accuracy for a single minibatch: 94.53%
Test set accuracy for a single minibatch: 91.41%


Epoch 0, Iteration 250
Train Set Loss: 15.11
Test Set Loss: 12.10
Train set accuracy for a single minibatch: 92.19%
Test set accuracy for a single minibatch: 94.53%


Ep